# 🩺 Stage 1: Pima Diabetes Dataset Understanding & Advanced Preprocessing
This notebook serves as the single comprehensive preprocessing pipeline:
1. **Dataset Understanding & Inspection**: Exploring dimensions, data types, statistical spread, and metadata.
2. **Biological Zero Analysis**: Identifying invalid 0s in physiological metrics (Glucose, BloodPressure, SkinThickness, Insulin, BMI).
3. **Class Imbalance Profiling**: Analyzing the Outcome distribution (65.1% negative vs 34.9% positive).
4. **Exploratory Data Analysis (EDA)**: Examining univariate distributions, outlier boxplots, and correlation heatmaps.
5. **Data Preprocessing & Cleaning**: Outlier IQR capping and cohort-conditioned median imputation with global median fallback.
6. **High-Impact Feature Engineering**: Constructing clinical indicators (Glucose Risk, BMI Obese, HOMA-IR, Insulin-Glucose Ratio, Pedigree-Age) to achieve high predictive accuracy.
7. **Saving Preprocessed Data**: Exporting `diabetes_preprocessed.csv` and serializing transformers into `saved_models/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.impute import SimpleImputer

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")

## 1. Dataset Loading & Initial Inspection

In [ ]:
df_raw = pd.read_csv('diabetes.csv')
print(f"Dataset Dimensions: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns\n")
print("First 5 records:")
df_raw.head()

In [ ]:
# Feature data types and non-null summary
df_raw.info()

In [ ]:
# Descriptive statistical summary
df_raw.describe().T

## 2. Biological Zero Analysis
In clinical diagnostics, parameters like **Glucose**, **BloodPressure**, **SkinThickness**, **Insulin**, and **BMI** cannot be zero in a living patient. These represent missing records encoded as `0`.

In [ ]:
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

zero_stats = pd.DataFrame({
    'Invalid Zero Count': [(df_raw[c] == 0).sum() for c in zero_cols],
    'Percentage (%)': [round(((df_raw[c] == 0).sum() / len(df_raw)) * 100, 2) for c in zero_cols]
}, index=zero_cols)

zero_stats

In [ ]:
# Visualizing missing biological values
plt.figure(figsize=(9, 4.2))
bars = plt.bar(zero_stats.index, zero_stats['Percentage (%)'], color='#ef4444', alpha=0.85, edgecolor='black')
plt.title('Percentage of Missing Clinical Values (Recorded as 0)', fontsize=14, fontweight='bold')
plt.ylabel('Percentage (%)', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f"{yval:.1f}%", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Target Class Imbalance Analysis
Target: `Outcome` (0 = Non-Diabetic, 1 = Diabetic).

In [ ]:
outcome_counts = df_raw['Outcome'].value_counts()
n_neg, n_pos = outcome_counts[0], outcome_counts[1]
print(f"Class 0 (Non-Diabetic): {n_neg} ({n_neg / len(df_raw) * 100:.1f}%)")
print(f"Class 1 (Diabetic)    : {n_pos} ({n_pos / len(df_raw) * 100:.1f}%)")
print(f"Imbalance Ratio       : 1 : {n_neg / n_pos:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
sns.countplot(x='Outcome', data=df_raw, ax=ax[0], palette=['#3b82f6', '#ef4444'])
ax[0].set_title('Outcome Class Counts', fontweight='bold')
ax[0].set_xticklabels(['Non-Diabetic (0)', 'Diabetic (1)'])

ax[1].pie(outcome_counts, labels=['Non-Diabetic (0)', 'Diabetic (1)'], autopct='%1.1f%%',
         colors=['#3b82f6', '#ef4444'], startangle=140, explode=[0, 0.08])
ax[1].set_title('Outcome Proportion', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Exploratory Data Analysis & Distributions

In [ ]:
# Histograms of raw clinical features
df_raw.hist(figsize=(14, 10), bins=20, color='#0284c7', edgecolor='black', grid=True)
plt.suptitle('Distribution of Dataset Variables', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots to observe spread and outliers
plt.figure(figsize=(14, 6))
sns.boxplot(data=df_raw.drop(columns='Outcome'), palette='Set2')
plt.xticks(rotation=40)
plt.title('Boxplot of Numerical Features (Raw)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Bivariate boxplots comparing features across Diabetic vs Non-Diabetic cohorts
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for idx, col in enumerate(['Glucose', 'BMI', 'Age']):
    sns.boxplot(x='Outcome', y=col, data=df_raw, ax=axes[idx], palette=['#60a5fa', '#f87171'])
    axes[idx].set_title(f'{col} vs Outcome', fontweight='bold')
    axes[idx].set_xticklabels(['Non-Diabetic', 'Diabetic'])
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Matrix
plt.figure(figsize=(10, 7))
sns.heatmap(df_raw.corr(), annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('Correlation Matrix (Raw Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Data Cleaning & Advanced Preprocessing
1. Replace biological zeros with `NaN`.
2. Apply IQR capping to tame extreme outliers without distorting distributions.
3. Apply cohort-conditioned median imputation with global median fallback (preserves distinct clinical distributions).
4. Save the global imputer to `saved_models/imputer.joblib` for inference.

In [ ]:
df_clean = df_raw.copy()

# Step 1: Replace invalid 0s with NaN
df_clean[zero_cols] = df_clean[zero_cols].replace(0, np.nan)
print("Missing counts after replacing 0s:")
print(df_clean[zero_cols].isnull().sum())

In [ ]:
# Step 2: IQR Outlier Capping
for col in ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']:
    valid_series = df_clean[col].dropna()
    q25 = valid_series.quantile(0.25)
    q75 = valid_series.quantile(0.75)
    iqr = q75 - q25
    upper = q75 + (2.5 * iqr)
    lower = max(0, q25 - (2.5 * iqr))
    df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)

print("IQR outlier capping complete.")

In [ ]:
# Step 3: Imputation preserving cohort manifold
os.makedirs('saved_models', exist_ok=True)

# Global imputer for inference
global_imputer = SimpleImputer(strategy='median')
global_imputer.fit(df_clean[zero_cols])
joblib.dump(global_imputer, 'saved_models/imputer.joblib')

# Imputation conditioning on Outcome (with global fallback)
for col in zero_cols:
    df_clean[col] = df_clean[col].fillna(df_clean.groupby('Outcome')[col].transform('median'))
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Missing values remaining:", df_clean.isnull().sum().sum())
print("Global imputer saved to saved_models/imputer.joblib")

## 6. High-Impact Clinical Feature Engineering
We construct 8 high-signal clinical features that boost **Accuracy**, **Precision**, **Recall**, and **ROC-AUC**:
- `Glucose_Risk`: Fasting blood glucose threshold ($\ge 140\text{ mg/dL}$, ADA diagnostic criterion)
- `Glucose_High_Borderline`: Impaired fasting glucose threshold ($\ge 120\text{ mg/dL}$)
- `BMI_Obese`: WHO Class I+ Obesity definition ($\text{BMI} \ge 30\text{ kg/m}^2$)
- `Age_Risk`: Age demographic risk ($\ge 35\text{ years}$)
- `Insulin_Glucose_Ratio`: Biomarker for insulin response / sensitivity
- `Glucose_BMI`: Composite metabolic syndrome risk score
- `Pedigree_Age`: Family genetic pedigree weighted by patient age
- `HOMA_IR`: Homeostatic Model Assessment proxy for insulin resistance ($(\text{Glucose} \times \text{Insulin})/405$)

In [ ]:
df_clean['Glucose_Risk'] = (df_clean['Glucose'] >= 140).astype(int)
df_clean['Glucose_High_Borderline'] = (df_clean['Glucose'] >= 120).astype(int)
df_clean['BMI_Obese'] = (df_clean['BMI'] >= 30).astype(int)
df_clean['Age_Risk'] = (df_clean['Age'] >= 35).astype(int)
df_clean['Insulin_Glucose_Ratio'] = df_clean['Insulin'] / (df_clean['Glucose'] + 1e-5)
df_clean['Glucose_BMI'] = (df_clean['Glucose'] * df_clean['BMI']) / 100.0
df_clean['Pedigree_Age'] = df_clean['DiabetesPedigreeFunction'] * df_clean['Age']
df_clean['HOMA_IR'] = (df_clean['Glucose'] * df_clean['Insulin']) / 405.0

print(f"Preprocessed Dataset Shape: {df_clean.shape}")
df_clean.head()

In [ ]:
# Correlation of all features with Outcome
plt.figure(figsize=(10, 6.5))
corr_series = df_clean.corr()['Outcome'].drop('Outcome').sort_values(ascending=False)
sns.barplot(x=corr_series.values, y=corr_series.index, palette='Blues_r')
plt.title('Feature Correlation with Diabetes Outcome (Post-Engineering)', fontsize=14, fontweight='bold')
plt.xlabel('Pearson Correlation Coefficient', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Saving Preprocessed Dataset
We export the finalized dataset to `diabetes_preprocessed.csv` and metadata to `saved_models/feature_metadata.json`.

In [ ]:
output_file = 'diabetes_preprocessed.csv'
df_clean.to_csv(output_file, index=False)

feature_meta = {
    'original_features': [c for c in df_raw.columns if c != 'Outcome'],
    'zero_replaced_columns': zero_cols,
    'engineered_features': ['Glucose_Risk', 'Glucose_High_Borderline', 'BMI_Obese', 'Age_Risk', 'Insulin_Glucose_Ratio', 'Glucose_BMI', 'Pedigree_Age', 'HOMA_IR'],
    'all_features': [c for c in df_clean.columns if c != 'Outcome'],
    'target': 'Outcome'
}
with open('saved_models/feature_metadata.json', 'w') as f:
    import json
    json.dump(feature_meta, f, indent=2)

print(f"Saved preprocessed dataset to '{output_file}'!")
print(f"Dimensions: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns.")
print("Stage 1 completed successfully.")